# 🔵 Customer Segmentation — RFM + K-Means Clustering

> **Dataset:** Online Retail (UCI Machine Learning Repository)  
> **Source:** https://archive.ics.uci.edu/dataset/352/online+retail  
> **Goal:** Use unsupervised machine learning (K-Means clustering) to discover natural customer segments from RFM data — without imposing manual rules — and translate each segment into a distinct marketing strategy.

---

## Why Clustering Over Manual Segmentation?

Manual RFM buckets (High/Mid/Low) are intuitive but arbitrary. K-Means lets the data speak: it finds groups of customers who are genuinely similar to each other and distinct from other groups.

| Approach | Strength | Weakness |
|---|---|---|
| Manual RFM tiers | Easy to explain | Thresholds are arbitrary |
| K-Means clustering | Data-driven, finds real patterns | Requires interpretation |
| **RFM + K-Means (this notebook)** | Best of both — clusters labelled by RFM profile | Slightly more complex |

---
## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# ── Palette ─────────────────────────────────────────────────
CHOCOLATE  = '#3d2314'
BROWN      = '#7a4f35'
CAMEL      = '#b89a74'
TERRACOTTA = '#c4694f'
SAND       = '#d9cdb8'
PARCHMENT  = '#ede5d4'
GREEN_OK   = '#5a8a5e'

SEGMENT_COLORS = [
    CHOCOLATE, TERRACOTTA, CAMEL, GREEN_OK,
    BROWN, '#4a4e69', '#708090', '#c4a14f'
]

plt.rcParams.update({
    'figure.facecolor':  PARCHMENT,
    'axes.facecolor':    '#f5f0e8',
    'axes.edgecolor':    SAND,
    'axes.labelcolor':   BROWN,
    'axes.titlecolor':   CHOCOLATE,
    'axes.titlesize':    13,
    'axes.titleweight':  'normal',
    'axes.labelsize':    10,
    'xtick.color':       BROWN,
    'ytick.color':       BROWN,
    'grid.color':        SAND,
    'grid.linestyle':    '--',
    'grid.alpha':        0.5,
    'font.family':       'serif',
    'text.color':        CHOCOLATE,
})

import os
os.makedirs('outputs', exist_ok=True)
print('Libraries loaded ✅')

---
## 2. Load & Clean Data

In [ ]:
df = pd.read_excel('data/online_retail.xlsx', dtype={'CustomerID': str})

df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
df = df.dropna(subset=['CustomerID'])
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
df['Revenue'] = df['Quantity'] * df['UnitPrice']
df = df[df['Country'] == 'United Kingdom'].copy()

print(f'Clean rows: {len(df):,} | Customers: {df.CustomerID.nunique():,}')

---
## 3. Build RFM Table

In [ ]:
snapshot = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg(
    Recency   = ('InvoiceDate', lambda x: (snapshot - x.max()).days),
    Frequency = ('InvoiceNo', 'nunique'),
    Monetary  = ('Revenue', 'sum'),
).reset_index()

# Remove extreme outliers (top 1%) for cleaner clustering
for col in ['Recency','Frequency','Monetary']:
    upper = rfm[col].quantile(0.99)
    rfm = rfm[rfm[col] <= upper]

print(f'Customers for clustering: {len(rfm):,}')
rfm.describe().round(2)

---
## 4. Find Optimal Number of Clusters

We use two methods:
- **Elbow method** — inertia drops steeply then levels off; the elbow is the optimal k
- **Silhouette score** — measures how well-separated clusters are (higher = better)

In [ ]:
# Scale features
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency','Frequency','Monetary']])

inertias   = []
silhouettes = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(rfm_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(rfm_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(list(K_range), inertias, marker='o', color=CHOCOLATE, lw=2)
ax.set_title('Elbow Method — Inertia vs K')
ax.set_xlabel('Number of Clusters (K)')
ax.set_ylabel('Inertia')
ax.grid(True, alpha=0.4)

ax2 = axes[1]
ax2.plot(list(K_range), silhouettes, marker='o', color=TERRACOTTA, lw=2)
best_k_sil = list(K_range)[silhouettes.index(max(silhouettes))]
ax2.axvline(best_k_sil, color=CAMEL, linestyle='--', lw=1.5,
            label=f'Best K={best_k_sil}')
ax2.set_title('Silhouette Score vs K')
ax2.set_xlabel('Number of Clusters (K)')
ax2.set_ylabel('Silhouette Score')
ax2.legend(framealpha=0.7)
ax2.grid(True, alpha=0.4)

plt.suptitle('Optimal K Selection', fontsize=14, color=CHOCOLATE)
plt.tight_layout()
plt.savefig('outputs/clustering_optimal_k.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Best K by Silhouette: {best_k_sil}')
print('Review elbow plot and choose K based on both methods.')

---
## 5. Fit Final K-Means Model

In [ ]:
# Set K — adjust based on elbow + silhouette plots above
K = 4

km_final = KMeans(n_clusters=K, random_state=42, n_init=10)
rfm['Cluster'] = km_final.fit_predict(rfm_scaled)

print(f'K-Means fitted with K={K} ✅')
print(f'Silhouette score: {silhouette_score(rfm_scaled, rfm["Cluster"]):.4f}')
print(f'\nCluster sizes:')
print(rfm['Cluster'].value_counts().sort_index())

---
## 6. Interpret & Label Segments

We profile each cluster by its mean RFM values and assign a business-meaningful name.

In [ ]:
cluster_profile = rfm.groupby('Cluster').agg(
    Customers   = ('CustomerID', 'count'),
    AvgRecency  = ('Recency', 'mean'),
    AvgFrequency= ('Frequency', 'mean'),
    AvgMonetary = ('Monetary', 'mean'),
    TotalRevenue= ('Monetary', 'sum'),
).round(1)

cluster_profile['RevenueShare'] = (
    cluster_profile['TotalRevenue'] / cluster_profile['TotalRevenue'].sum()
).map('{:.1%}'.format)

print(cluster_profile)
print()
print('── Assign segment labels based on the profiles above ──')
print('Low Recency + High Freq + High Monetary  → Champions')
print('Low Recency + Mid Freq  + Mid Monetary   → Loyal Customers')
print('High Recency + Low Freq + Any Monetary   → At Risk / Dormant')
print('Mid Recency + Low Freq  + Low Monetary   → New / Occasional')

In [ ]:
# ── Assign labels based on cluster profiles ───────────────────
# Review the profile above and map cluster numbers to segment names
# This mapping will depend on your actual cluster output — adjust accordingly

# Sort clusters by AvgMonetary descending to auto-assign labels
sorted_clusters = cluster_profile['AvgMonetary'].sort_values(ascending=False).index.tolist()

segment_labels = {
    sorted_clusters[0]: 'Champions',
    sorted_clusters[1]: 'Loyal Customers',
    sorted_clusters[2]: 'At Risk',
    sorted_clusters[3]: 'Occasional Buyers',
}

rfm['Segment'] = rfm['Cluster'].map(segment_labels)

print('Segment assignment:')
print(rfm['Segment'].value_counts())

---
## 7. Visualisations

### 7a. Cluster Scatter Plots — 2D Projections

In [ ]:
segments = rfm['Segment'].unique()
seg_colors = dict(zip(segments, SEGMENT_COLORS[:len(segments)]))

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

pairs = [
    ('Recency', 'Monetary',  'Recency (days)', 'Total Spend (£)'),
    ('Frequency', 'Monetary','Orders',          'Total Spend (£)'),
    ('Recency', 'Frequency', 'Recency (days)', 'Orders'),
]

for ax, (x, y, xlabel, ylabel) in zip(axes, pairs):
    for seg in segments:
        sub = rfm[rfm['Segment'] == seg]
        ax.scatter(sub[x], sub[y], alpha=0.35, s=18,
                   color=seg_colors[seg], label=seg, edgecolors='none')
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(f'{xlabel} vs {ylabel}')
    ax.grid(True, alpha=0.4)
    if x == 'Recency':
        ax.legend(fontsize=7, framealpha=0.7, markerscale=1.5)

plt.suptitle('Customer Segments — RFM Scatter Plots', fontsize=14, color=CHOCOLATE)
plt.tight_layout()
plt.savefig('outputs/clustering_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

### 7b. PCA — 2D Projection of All 3 Dimensions

In [ ]:
pca = PCA(n_components=2, random_state=42)
rfm_pca = pca.fit_transform(rfm_scaled)
rfm['PC1'] = rfm_pca[:,0]
rfm['PC2'] = rfm_pca[:,1]

fig, ax = plt.subplots(figsize=(11, 7))

for seg in segments:
    sub = rfm[rfm['Segment'] == seg]
    ax.scatter(sub['PC1'], sub['PC2'], alpha=0.4, s=22,
               color=seg_colors[seg], label=seg, edgecolors='none')

# Centroids
centroids_pca = pca.transform(km_final.cluster_centers_)
for i, (cx, cy) in enumerate(centroids_pca):
    seg = segment_labels[i]
    ax.scatter(cx, cy, s=180, color=seg_colors[seg],
               marker='*', edgecolors=CHOCOLATE, linewidths=1, zorder=5)

var_explained = pca.explained_variance_ratio_ * 100
ax.set_xlabel(f'PC1 ({var_explained[0]:.1f}% variance explained)')
ax.set_ylabel(f'PC2 ({var_explained[1]:.1f}% variance explained)')
ax.set_title('Customer Segments — PCA Projection (★ = cluster centroid)')
ax.legend(title='Segment', framealpha=0.7)
ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('outputs/clustering_pca.png', dpi=150, bbox_inches='tight')
plt.show()

### 7c. Segment Profile — Snake Plot

In [ ]:
# Normalise RFM values to 0-1 scale for comparison
rfm_norm = rfm.copy()
for col in ['Recency','Frequency','Monetary']:
    rfm_norm[col] = (rfm[col] - rfm[col].min()) / (rfm[col].max() - rfm[col].min())

# Note: for Recency, lower = better, so invert
rfm_norm['Recency'] = 1 - rfm_norm['Recency']

snake_data = rfm_norm.groupby('Segment')[['Recency','Frequency','Monetary']].mean()

fig, ax = plt.subplots(figsize=(10, 5))

for seg in segments:
    ax.plot(['Recency\n(inverted)','Frequency','Monetary'],
            snake_data.loc[seg],
            marker='o', lw=2.5, markersize=8,
            color=seg_colors[seg], label=seg)

ax.set_ylim(0, 1)
ax.set_ylabel('Normalised Score (higher = better)')
ax.set_title('Segment Snake Plot — Normalised RFM Profile')
ax.legend(title='Segment', framealpha=0.7)
ax.grid(True, alpha=0.4)
ax.axhline(0.5, color=SAND, linestyle='--', lw=1)

plt.tight_layout()
plt.savefig('outputs/clustering_snake_plot.png', dpi=150, bbox_inches='tight')
plt.show()

### 7d. Revenue & Customer Share by Segment

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

seg_summary = rfm.groupby('Segment').agg(
    Customers    = ('CustomerID', 'count'),
    TotalRevenue = ('Monetary', 'sum')
)

colors_list = [seg_colors[s] for s in seg_summary.index]

for ax, col, title in zip(axes,
    ['Customers','TotalRevenue'],
    ['Customer Share by Segment','Revenue Share by Segment']):
    wedges, texts, autotexts = ax.pie(
        seg_summary[col], labels=seg_summary.index,
        autopct='%1.1f%%', colors=colors_list,
        startangle=90,
        wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2),
        pctdistance=0.75
    )
    for at in autotexts: at.set_fontsize(8)
    ax.set_title(title)

plt.suptitle('Segment Size & Revenue Contribution', fontsize=14, color=CHOCOLATE)
plt.tight_layout()
plt.savefig('outputs/clustering_revenue_share.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Marketing Strategy per Segment

In [ ]:
strategy = {
    'Champions': {
        'Profile':    'Recent, frequent, high spend',
        'Goal':       'Retain & grow',
        'Tactics':    'VIP programme, early access, personalised upsells, loyalty rewards',
        'Channels':   'Email (personalised), direct mail, app push',
        'Budget':     'High — highest ROI on retention spend',
        'Risk':       'Low churn risk but high cost if lost',
    },
    'Loyal Customers': {
        'Profile':    'Regular buyers, mid–high spend, moderate recency',
        'Goal':       'Increase AOV & upgrade to Champions',
        'Tactics':    'Cross-sell complementary products, bundle offers, tier upgrade nudges',
        'Channels':   'Email, retargeting ads',
        'Budget':     'Medium',
        'Risk':       'Moderate — at risk if ignored',
    },
    'At Risk': {
        'Profile':    'Previously active, now dormant — high recency days',
        'Goal':       'Re-activate before full churn',
        'Tactics':    'Win-back email sequence, time-limited discount, "we miss you" message',
        'Channels':   'Email, SMS (if consented)',
        'Budget':     'Medium — prioritise high-spend at-risk customers',
        'Risk':       'High churn risk — act within 30 days',
    },
    'Occasional Buyers': {
        'Profile':    'Low frequency, low spend, may be new or price-sensitive',
        'Goal':       'Activate & move to Loyal',
        'Tactics':    'Onboarding sequence, first-repeat-purchase incentive, educational content',
        'Channels':   'Email, paid social',
        'Budget':     'Low — high volume, uncertain LTV',
        'Risk':       'High natural churn but low cost to retain',
    },
}

strategy_df = pd.DataFrame(strategy).T
strategy_df.index.name = 'Segment'
strategy_df

---
## 9. Export

In [ ]:
rfm[['CustomerID','Recency','Frequency','Monetary','Cluster','Segment']]\
    .sort_values('Monetary', ascending=False)\
    .to_csv('outputs/customer_segments.csv', index=False)

strategy_df.to_csv('outputs/segment_strategy.csv')

print('Exported:')
print('  → outputs/customer_segments.csv ✅')
print('  → outputs/segment_strategy.csv ✅')

---
## 10. Key Findings & Recommendations

---

### 🔍 Findings

| # | Finding |
|---|---|
| 1 | **K-Means identifies 4 natural customer groups** — the data-driven segments align well with intuitive RFM interpretation |
| 2 | **Champions are few but dominate revenue** — consistent with LTV and Cohort analyses in this portfolio |
| 3 | **At-Risk segment is significant in size** — a large number of previously active customers have gone quiet |
| 4 | **Occasional Buyers are the largest group by volume** — most customers make very few purchases, confirming the activation challenge |
| 5 | **Snake plot reveals clear differentiation** — clusters are genuinely distinct across all three RFM dimensions |

---

### 💡 Recommendations

**1. Treat segments as dynamic, not static**  
Re-run clustering monthly. A customer who was Occasional last month may be Loyal this month. Automating segment re-assignment in your CRM ensures campaigns are always relevant.

**2. Connect clusters to the Churn Model**  
Overlay the churn probability scores from Case 06 onto each segment. At-Risk customers with high churn probability should be your immediate priority — they're both behaviourally dormant and statistically likely to leave.

**3. Don't spend equally across segments**  
Retention spend on Champions has the highest ROI. Acquisition spend targeting Occasional Buyers has the lowest. Allocate budget proportionally to segment revenue contribution.

**4. Use segments to personalise, not just target**  
Different segments respond to different messages. Champions respond to exclusivity; At-Risk customers respond to urgency; Occasional Buyers respond to social proof and first-purchase incentives.

**5. Test K=3 vs K=4 vs K=5 in your live CRM**  
More clusters = more specificity but more complexity for the marketing team to manage. Run a pilot with 3 segments first; add granularity only once you have the operational capacity.

---

*Analysis by Danai Avratoglou | Dataset: UCI Online Retail | Tools: Python, scikit-learn (K-Means, PCA), pandas, matplotlib*